# EndoPredict: Diabetes Risk Prediction Training Pipeline

This notebook contains the complete machine learning training pipeline for predicting diabetes using the Kaggle Diabetes Prediction Dataset. It includes:
1. Data Loading & Inspection
2. Preprocessing & Feature Engineering
3. Random Forest Model Training
4. Pipeline Evaluation & Metrics
5. Saving the Trained Model Artifact

In [1]:
import os
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, roc_auc_score
import joblib

# Alias pd.csv to pd.read_csv as per requirements
pd.csv = pd.read_csv

## 1. Load Dataset
We load the `diabetes_prediction_dataset.csv` using the custom `pd.csv` method.

In [2]:
dataset_path = "diabetes_prediction_dataset.csv"
df = pd.csv(dataset_path)
print(f"Dataset loaded successfully. Shape: {df.shape}")

Dataset loaded successfully. Shape: (100000, 9)


## 2. Data Inspection

In [3]:
print("Dataset Info:")
print(df.info())

print("\nTarget Class Distribution (diabetes):")
print(df['diabetes'].value_counts(normalize=True))

Dataset Info:


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100000 entries, 0 to 99999
Data columns (total 9 columns):
 #   Column               Non-Null Count   Dtype  
---  ------               --------------   -----  
 0   gender               100000 non-null  object 
 1   age                  100000 non-null  float64
 2   hypertension         100000 non-null  int64  
 3   heart_disease        100000 non-null  int64  
 4   smoking_history      100000 non-null  object 
 5   bmi                  100000 non-null  float64
 6   HbA1c_level          100000 non-null  float64
 7   blood_glucose_level  100000 non-null  int64  
 8   diabetes             100000 non-null  int64  
dtypes: float64(3), int64(4), object(2)
memory usage: 6.9+ MB
None

Target Class Distribution (diabetes):
diabetes
0    0.915
1    0.085
Name: proportion, dtype: float64


## 3. Data Splitting & Feature Separation

In [4]:
X = df.drop(columns=['diabetes'])
y = df['diabetes']

# Define column configurations
num_cols = ['age', 'bmi', 'HbA1c_level', 'blood_glucose_level']
cat_cols = ['gender', 'smoking_history']
bin_cols = ['hypertension', 'heart_disease']

## 4. Preprocessing & Classifier Pipeline Setup

In [5]:
# Preprocessing pipeline
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), num_cols),
        ('cat', OneHotEncoder(drop='first', handle_unknown='ignore'), cat_cols)
    ],
    remainder='passthrough'
)

# Random Forest classifier with class balancing to manage target skew
pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', RandomForestClassifier(
        n_estimators=100,
        max_depth=12,
        class_weight='balanced',
        random_state=42
    ))
])

## 5. Model Training

In [6]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print("Training model pipeline...")
pipeline.fit(X_train, y_train)
print("Training completed successfully.")

Training model pipeline...


Training completed successfully.


## 6. Evaluation

In [7]:
y_pred = pipeline.predict(X_test)
y_pred_proba = pipeline.predict_proba(X_test)[:, 1]

print(f"Accuracy: {accuracy_score(y_test, y_pred):.4f}")
print(f"ROC-AUC Score: {roc_auc_score(y_test, y_pred_proba):.4f}\n")

print("Classification Report:")
print(classification_report(y_test, y_pred))

print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred))

Accuracy: 0.9182
ROC-AUC Score: 0.9764

Classification Report:
              precision    recall  f1-score   support

           0       0.99      0.92      0.95     18300
           1       0.51      0.89      0.65      1700

    accuracy                           0.92     20000
   macro avg       0.75      0.90      0.80     20000
weighted avg       0.95      0.92      0.93     20000

Confusion Matrix:
[[16853  1447]
 [  189  1511]]


### Feature Importances

In [8]:
try:
    onehot_features = pipeline.named_steps['preprocessor'].named_transformers_['cat'].get_feature_names_out(cat_cols)
    feature_names = num_cols + list(onehot_features) + bin_cols
    importances = pipeline.named_steps['classifier'].feature_importances_
    
    indices = np.argsort(importances)[::-1]
    print("Feature Importances:")
    for i in range(len(feature_names)):
        print(f"{i+1}. {feature_names[indices[i]]}: {importances[indices[i]]:.4f}")
except Exception as e:
    print(f"Could not extract feature importances: {e}")

Feature Importances:
1. HbA1c_level: 0.4207
2. blood_glucose_level: 0.2807
3. age: 0.1549
4. bmi: 0.0785
5. hypertension: 0.0297
6. heart_disease: 0.0190
7. smoking_history_former: 0.0074
8. gender_Male: 0.0036
9. smoking_history_never: 0.0019
10. smoking_history_current: 0.0014
11. smoking_history_not current: 0.0012
12. smoking_history_ever: 0.0010
13. gender_Other: 0.0000


## 7. Save Model Pipeline

In [9]:
model_filename = "diabetes_model_pipeline.joblib"
print(f"Saving pipeline to {model_filename}...")
joblib.dump(pipeline, model_filename)
print("Saved successfully.")

Saving pipeline to diabetes_model_pipeline.joblib...


Saved successfully.
